# Practice Lab: Neural Networks for Handwritten Digit Recognition (Binary) — Digits "3" vs "8"

This notebook is a **remixed version** of the classic Coursera *"Neural Networks for Handwritten Digit Recognition, Binary"* practice lab.

Instead of classifying digits **0** and **1**, this version builds a neural network that classifies two harder-to-distinguish digits: **3** and **8**, using the built-in `scikit-learn` digits dataset (8x8 pixel images, no external files to download — everything runs standalone).

The topic and structure are identical to the original lab, so it's great for reinforcing the same concepts with fresh data:

## Outline
- [1 - Packages](#1)
- [2 - Neural Networks](#2)
  - [2.1 Problem Statement](#2.1)
  - [2.2 Dataset](#2.2)
  - [2.3 Model Representation](#2.3)
  - [2.4 TensorFlow Model Implementation](#2.4)
  - [2.5 NumPy Model Implementation (Forward Prop in NumPy)](#2.5)
  - [2.6 Vectorized NumPy Model Implementation](#2.6)
  - [2.7 Congratulations!](#2.7)
  - [2.8 NumPy Broadcasting Tutorial (Optional)](#2.8)


<a name="1"></a>
## 1 - Packages

We'll use:
- [numpy](https://numpy.org/) — fundamental package for scientific computing.
- [matplotlib](https://matplotlib.org/) — plotting.
- [tensorflow](https://www.tensorflow.org/) — building/training the neural network.
- [scikit-learn](https://scikit-learn.org/) — only used to load the built-in `digits` dataset (no downloads required, it ships with the library).


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

%matplotlib inline

import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)
tf.autograph.set_verbosity(0)

np.random.seed(1)
tf.random.set_seed(1)

print("TensorFlow version:", tf.__version__)


**TensorFlow and Keras**
TensorFlow is a machine learning framework developed by Google. Keras is its high level, layer-centric API for building models. This notebook uses the Keras `Sequential` API, exactly like the original lab.


<a name="2"></a>
## 2 - Neural Networks

<a name="2.1"></a>
### 2.1 Problem Statement

We will build a neural network to recognize two handwritten digits: **3** and **8**. This is a **binary classification** task — same idea as recognizing 0 vs 1 in the original lab, but 3 vs 8 is a noticeably harder pair since the digits look more visually similar.

<a name="2.2"></a>
### 2.2 Dataset

We use `sklearn.datasets.load_digits()`, which contains **1,797** 8x8 grayscale images of handwritten digits (0-9), each pixel valued 0-16.

- We will filter the dataset down to only the examples labeled `3` or `8`.
- Each image is 8 pixels x 8 pixels, "unrolled" into a 64-dimensional vector (much like the 400-dim vectors from 20x20 images in the original lab, just smaller).
- We'll relabel `3 -> 0` and `8 -> 1` so the network learns a clean binary target, exactly mirroring how the original lab used 0/1 labels directly.


In [ ]:
# Load the full digits dataset
digits = load_digits()
all_images = digits.images        # shape (1797, 8, 8)
all_data = digits.data            # shape (1797, 64), already unrolled
all_labels = digits.target        # shape (1797,)

# Keep only digits 3 and 8
mask = (all_labels == 3) | (all_labels == 8)

X = all_data[mask]
raw_labels = all_labels[mask]

# Normalize pixel values to [0, 1] (original values are 0-16)
X = X / 16.0

# Relabel: 3 -> 0, 8 -> 1  (mirrors the 0/1 style label used in the original lab)
y = np.where(raw_labels == 8, 1, 0).reshape(-1, 1)

print("Loaded dataset restricted to digits 3 and 8.")


#### 2.2.1 View the variables
Let's get familiar with the dataset by printing a couple of elements.


In [ ]:
print('The first element of X is:\n', X[0])


In [ ]:
print('The first element of y is:', y[0, 0])
print('The last element of y is: ', y[-1, 0])


#### 2.2.2 Check the dimensions of your variables


In [ ]:
print('The shape of X is: ' + str(X.shape))
print('The shape of y is: ' + str(y.shape))
print('Number of "3" examples (label 0):', np.sum(y == 0))
print('Number of "8" examples (label 1):', np.sum(y == 1))


#### 2.2.3 Visualizing the Data

We'll randomly select 64 rows from `X`, reshape each back into an 8x8 image, and display them together with their true label above each image.


In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

m, n = X.shape

fig, axes = plt.subplots(8, 8, figsize=(8, 8))
fig.tight_layout(pad=0.1)

for i, ax in enumerate(axes.flat):
    # Select a random index
    random_index = np.random.randint(m)

    # Reshape the row back into an 8x8 image
    X_random_reshaped = X[random_index].reshape((8, 8))

    # Display the image
    ax.imshow(X_random_reshaped, cmap='gray')

    # Display the true digit (3 or 8) above the image
    true_digit = 3 if y[random_index, 0] == 0 else 8
    ax.set_title(str(true_digit))
    ax.set_axis_off()

fig.suptitle("Sample digits (3s and 8s)", fontsize=14)
plt.show()


<a name="2.3"></a>
### 2.3 Model Representation

We'll use the same three-layer network shape as the original lab, just with a smaller input layer since our images are 8x8 = 64 pixels instead of 20x20 = 400 pixels:

- **Layer 1:** 25 units, sigmoid activation
- **Layer 2:** 15 units, sigmoid activation
- **Layer 3:** 1 unit, sigmoid activation (outputs a probability that the digit is an "8")

If a layer has $s_{in}$ input units and $s_{out}$ output units, then:
- $W$ has shape $(s_{in}, s_{out})$
- $b$ has shape $(s_{out},)$

So the shapes here are:
- Layer 1: `W1` shape (64, 25), `b1` shape (25,)
- Layer 2: `W2` shape (25, 15), `b2` shape (15,)
- Layer 3: `W3` shape (15, 1), `b3` shape (1,)


<a name="2.4"></a>
### 2.4 TensorFlow Model Implementation

TensorFlow/Keras models are built layer by layer. We specify each layer's *output* dimension; the *input* dimension is inferred automatically from the previous layer (or from `tf.keras.Input` for the first layer).


In [ ]:
model = Sequential(
    [
        tf.keras.Input(shape=(64,)),          # input size = 8 x 8 pixels unrolled
        Dense(units=25, activation='sigmoid', name='L1'),
        Dense(units=15, activation='sigmoid', name='L2'),
        Dense(units=1,  activation='sigmoid', name='L3'),
    ],
    name="digit_3_vs_8_model"
)

model.summary()


**Expected shape of the summary**
```
Model: "digit_3_vs_8_model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #
=================================================================
 L1 (Dense)                  (None, 25)                1625
 L2 (Dense)                  (None, 15)                390
 L3 (Dense)                  (None, 1)                 16
=================================================================
Total params: 2031
```


The parameter counts correspond to the number of elements in the weight and bias arrays, as shown below.


In [ ]:
L1_num_params = 64 * 25 + 25   # W1 parameters + b1 parameters
L2_num_params = 25 * 15 + 15   # W2 parameters + b2 parameters
L3_num_params = 15 * 1 + 1     # W3 parameters + b3 parameters
print("L1 params =", L1_num_params, ", L2 params =", L2_num_params, ", L3 params =", L3_num_params)


We can examine the layers directly with `model.layers`, and extract weights with `layer.get_weights()`.


In [ ]:
[layer1, layer2, layer3] = model.layers


In [ ]:
#### Examine weight shapes (before training, these are randomly initialized)
W1, b1 = layer1.get_weights()
W2, b2 = layer2.get_weights()
W3, b3 = layer3.get_weights()
print(f"W1 shape = {W1.shape}, b1 shape = {b1.shape}")
print(f"W2 shape = {W2.shape}, b2 shape = {b2.shape}")
print(f"W3 shape = {W3.shape}, b3 shape = {b3.shape}")


**Expected Output**
```
W1 shape = (64, 25), b1 shape = (25,)
W2 shape = (25, 15), b2 shape = (15,)
W3 shape = (15, 1), b3 shape = (1,)
```


Now let's compile the model with a loss function and optimizer, then run gradient descent to fit the weights to the training data.


In [ ]:
model.compile(
    loss=tf.keras.losses.BinaryCrossentropy(),
    optimizer=tf.keras.optimizers.Adam(0.001),
)

history = model.fit(
    X, y,
    epochs=40,
    verbose=0
)

print("Final training loss:", history.history['loss'][-1])


In [ ]:
plt.plot(history.history['loss'])
plt.title("Training loss over epochs")
plt.xlabel("Epoch")
plt.ylabel("Binary cross-entropy loss")
plt.show()


To make a prediction, use [Keras `predict`](https://www.tensorflow.org/api_docs/python/tf/keras/Model). The input must be 2D, so a single example is reshaped accordingly.


In [ ]:
idx_three = np.where(y[:, 0] == 0)[0][0]
idx_eight = np.where(y[:, 0] == 1)[0][0]

prediction = model.predict(X[idx_three].reshape(1, 64), verbose=0)
print(f"predicting a 3:  {prediction}")

prediction = model.predict(X[idx_eight].reshape(1, 64), verbose=0)
print(f"predicting an 8: {prediction}")


The output is a probability that the digit is an "8" (label 1). We compare it against a 0.5 threshold to get a hard prediction, just like logistic regression.


In [ ]:
if prediction >= 0.5:
    yhat = 1
else:
    yhat = 0
print(f"prediction after threshold: {yhat}")


Let's compare predictions vs. true labels for a random sample of 64 digits.


In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

m, n = X.shape

fig, axes = plt.subplots(8, 8, figsize=(8, 8))
fig.tight_layout(pad=0.1, rect=[0, 0.03, 1, 0.92])

for i, ax in enumerate(axes.flat):
    random_index = np.random.randint(m)

    X_random_reshaped = X[random_index].reshape((8, 8))
    ax.imshow(X_random_reshaped, cmap='gray')

    prediction = model.predict(X[random_index].reshape(1, 64), verbose=0)
    yhat = 1 if prediction >= 0.5 else 0

    true_digit = 3 if y[random_index, 0] == 0 else 8
    pred_digit = 3 if yhat == 0 else 8

    ax.set_title(f"{true_digit},{pred_digit}")
    ax.set_axis_off()

fig.suptitle("Label, yhat (as digits)", fontsize=16)
plt.show()


<a name="2.5"></a>
### 2.5 NumPy Model Implementation (Forward Prop in NumPy)

Just like the original lab, let's build our own dense layer from scratch with NumPy, then chain three of them together to reproduce the TensorFlow model's forward pass manually.

First, we define our own `sigmoid` function (the original lab imported this from a helper file — here we define it ourselves for a fully self-contained notebook).


In [ ]:
def sigmoid(z):
    '''
    Numerically stable sigmoid activation function.
    Args:
      z (ndarray): input array (any shape)
    Returns:
      ndarray: sigmoid(z), same shape as z
    '''
    return 1.0 / (1.0 + np.exp(-z))


Now build a dense layer subroutine. For each unit `j` in the layer, we take the dot product of the weights for that unit (`W[:, j]`) with the input, add the bias `b[j]`, and apply the activation function `g`.


In [ ]:
def my_dense(a_in, W, b, g):
    '''
    Computes a dense layer for a SINGLE example.
    Args:
      a_in (ndarray (n,))  : Data, 1 example
      W    (ndarray (n,j)) : Weight matrix, n features per unit, j units
      b    (ndarray (j,))  : bias vector, j units
      g    : activation function (e.g. sigmoid)
    Returns:
      a_out (ndarray (j,)) : activations for the j units
    '''
    units = W.shape[1]
    a_out = np.zeros(units)
    for j in range(units):
        w = W[:, j]
        z = np.dot(w, a_in) + b[j]
        a_out[j] = g(z)
    return a_out


In [ ]:
# Quick check against a tiny hand-worked example
x_tst = 0.1 * np.arange(1, 3, 1).reshape(2,)      # (2 features)
W_tst = 0.1 * np.arange(1, 7, 1).reshape(2, 3)    # (2 input features, 3 output units)
b_tst = 0.1 * np.arange(1, 4, 1).reshape(3,)      # (3 units)
A_tst = my_dense(x_tst, W_tst, b_tst, sigmoid)
print(A_tst)


**Expected Output**
```
[0.54735762 0.57932425 0.61063923]
```


The following cell chains three `my_dense` layers into a full forward pass, mirroring the TensorFlow model architecture.


In [ ]:
def my_sequential(x, W1, b1, W2, b2, W3, b3):
    a1 = my_dense(x,  W1, b1, sigmoid)
    a2 = my_dense(a1, W2, b2, sigmoid)
    a3 = my_dense(a2, W3, b3, sigmoid)
    return a3


We can copy the *trained* weights and biases straight out of the fitted TensorFlow model.


In [ ]:
W1_tmp, b1_tmp = layer1.get_weights()
W2_tmp, b2_tmp = layer2.get_weights()
W3_tmp, b3_tmp = layer3.get_weights()


In [ ]:
prediction = my_sequential(X[idx_three], W1_tmp, b1_tmp, W2_tmp, b2_tmp, W3_tmp, b3_tmp)
yhat = 1 if prediction[0] >= 0.5 else 0
print("yhat =", yhat, " label =", y[idx_three, 0])

prediction = my_sequential(X[idx_eight], W1_tmp, b1_tmp, W2_tmp, b2_tmp, W3_tmp, b3_tmp)
yhat = 1 if prediction[0] >= 0.5 else 0
print("yhat =", yhat, " label =", y[idx_eight, 0])


Let's compare predictions from the NumPy model and the TensorFlow model side by side.


In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

m, n = X.shape

fig, axes = plt.subplots(8, 8, figsize=(8, 8))
fig.tight_layout(pad=0.1, rect=[0, 0.03, 1, 0.92])

for i, ax in enumerate(axes.flat):
    random_index = np.random.randint(m)

    X_random_reshaped = X[random_index].reshape((8, 8))
    ax.imshow(X_random_reshaped, cmap='gray')

    my_prediction = my_sequential(X[random_index], W1_tmp, b1_tmp, W2_tmp, b2_tmp, W3_tmp, b3_tmp)
    my_yhat = int(my_prediction[0] >= 0.5)

    tf_prediction = model.predict(X[random_index].reshape(1, 64), verbose=0)
    tf_yhat = int(tf_prediction[0, 0] >= 0.5)

    true_digit = 3 if y[random_index, 0] == 0 else 8
    ax.set_title(f"{true_digit},{tf_yhat},{my_yhat}")
    ax.set_axis_off()

fig.suptitle("Label, yhat Tensorflow, yhat Numpy", fontsize=16)
plt.show()


<a name="2.6"></a>
### 2.6 Vectorized NumPy Model Implementation

Looping over every unit and every example is slow. We can speed this up dramatically using matrix operations. First, for a *single* example:

$$z_1 = x^T W + b$$

Then, for an entire *batch* of examples $\mathbf{X}$ at once:

$$\mathbf{Z} = \mathbf{X}\mathbf{W} + \mathbf{b}$$

This uses NumPy broadcasting to expand $\mathbf{b}$ across all $m$ rows (explained in the optional tutorial at the end of this notebook).


In [ ]:
x = X[idx_three].reshape(-1, 1)     # column vector (64, 1)
z1 = np.matmul(x.T, W1) + b1        # (1,64)(64,25) = (1,25)
a1 = sigmoid(z1)
print(a1.shape)


Now let's build a vectorized dense-layer function that processes an entire batch of examples in one shot using `np.matmul()`.


In [ ]:
def my_dense_v(A_in, W, b, g):
    '''
    Computes a dense layer for a BATCH of examples.
    Args:
      A_in (ndarray (m,n)) : Data, m examples, n features each
      W    (ndarray (n,j)) : Weight matrix, n features per unit, j units
      b    (ndarray (1,j)) : bias vector, j units
      g    : activation function (e.g. sigmoid)
    Returns:
      A_out (ndarray (m,j)) : m examples, j units
    '''
    Z = np.matmul(A_in, W) + b
    A_out = g(Z)
    return A_out


In [ ]:
X_tst = 0.1 * np.arange(1, 9, 1).reshape(4, 2)   # (4 examples, 2 features)
W_tst = 0.1 * np.arange(1, 7, 1).reshape(2, 3)   # (2 input features, 3 output units)
b_tst = 0.1 * np.arange(1, 4, 1).reshape(1, 3)   # (1, 3 units)
A_tst = my_dense_v(X_tst, W_tst, b_tst, sigmoid)
print(A_tst)


**Expected Output**
```
[[0.54735762 0.57932425 0.61063923]
 [0.57199613 0.61301418 0.65248946]
 [0.5962827  0.64565631 0.6921095 ]
 [0.62010643 0.67699586 0.72908792]]
```


Chain three vectorized dense layers into a full batch forward pass.


In [ ]:
def my_sequential_v(X_batch, W1, b1, W2, b2, W3, b3):
    A1 = my_dense_v(X_batch, W1, b1, sigmoid)
    A2 = my_dense_v(A1,      W2, b2, sigmoid)
    A3 = my_dense_v(A2,      W3, b3, sigmoid)
    return A3


In [ ]:
W1_tmp, b1_tmp = layer1.get_weights()
W2_tmp, b2_tmp = layer2.get_weights()
W3_tmp, b3_tmp = layer3.get_weights()


Now predict on *all* examples at once — note the shape of the output.


In [ ]:
Prediction = my_sequential_v(X, W1_tmp, b1_tmp, W2_tmp, b2_tmp, W3_tmp, b3_tmp)
Prediction.shape


In [ ]:
Yhat = (Prediction >= 0.5).astype(int)
print("predict a 3 (label 0):", Yhat[idx_three], " predict an 8 (label 1):", Yhat[idx_eight])


Let's measure overall accuracy and visualize a random sample of predictions.


In [ ]:
accuracy = np.mean(Yhat == y)
print(f"Overall accuracy on the full dataset: {accuracy * 100:.2f}%")


In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

m, n = X.shape

fig, axes = plt.subplots(8, 8, figsize=(8, 8))
fig.tight_layout(pad=0.1, rect=[0, 0.03, 1, 0.92])

for i, ax in enumerate(axes.flat):
    random_index = np.random.randint(m)

    X_random_reshaped = X[random_index].reshape((8, 8))
    ax.imshow(X_random_reshaped, cmap='gray')

    true_digit = 3 if y[random_index, 0] == 0 else 8
    pred_digit = 3 if Yhat[random_index, 0] == 0 else 8
    ax.set_title(f"{true_digit}, {pred_digit}")
    ax.set_axis_off()

fig.suptitle("Label, Yhat", fontsize=16)
plt.show()


You can also inspect a misclassified example, if any exist.


In [ ]:
errors = np.where(y != Yhat)[0]
if len(errors) > 0:
    random_index = errors[0]
    X_random_reshaped = X[random_index].reshape((8, 8))
    true_digit = 3 if y[random_index, 0] == 0 else 8
    pred_digit = 3 if Yhat[random_index, 0] == 0 else 8

    fig = plt.figure(figsize=(2, 2))
    plt.imshow(X_random_reshaped, cmap='gray')
    plt.title(f"true={true_digit}, predicted={pred_digit}")
    plt.axis('off')
    plt.show()
else:
    print("No misclassified examples found in this run!")


<a name="2.7"></a>
### 2.7 Congratulations!

You've successfully built and used a neural network — trained in TensorFlow, and re-implemented from scratch in plain NumPy (both looped and vectorized versions) — this time to tell handwritten **3s** apart from **8s**.


<a name="2.8"></a>
### 2.8 NumPy Broadcasting Tutorial (Optional)

In the last section, $\mathbf{Z} = \mathbf{X}\mathbf{W} + \mathbf{b}$ relied on NumPy broadcasting to expand the vector $\mathbf{b}$. Here's a quick refresher, unchanged in spirit from the original lab.

$\mathbf{X}\mathbf{W}$ is a matrix-matrix operation with dimensions $(m, j_1)(j_1, j_2)$, producing a matrix of dimension $(m, j_2)$. We then add a vector $\mathbf{b}$ of dimension $(1, j_2)$. NumPy "stretches" (broadcasts) $\mathbf{b}$ to shape $(m, j_2)$ automatically so the addition makes sense element-wise.

**Broadcasting rule:** comparing shapes from the rightmost dimension leftward, two dimensions are compatible when they are equal, or one of them is 1. If neither condition holds, NumPy raises a `ValueError`.

Try to guess the resulting shape before running each cell below.


In [ ]:
a = np.array([1, 2, 3]).reshape(-1, 1)  # (3,1)
b = 5
print(f"(a + b).shape: {(a + b).shape}, \na + b = \n{a + b}")


This applies to all element-wise operations, not just addition:


In [ ]:
a = np.array([1, 2, 3]).reshape(-1, 1)  # (3,1)
b = 5
print(f"(a * b).shape: {(a * b).shape}, \na * b = \n{a * b}")


Row + column broadcasting example:


In [ ]:
a = np.array([1, 2, 3, 4]).reshape(-1, 1)   # (4,1)
b = np.array([1, 2, 3]).reshape(1, -1)      # (1,3)
print(a)
print(b)
print(f"(a + b).shape: {(a + b).shape}, \na + b = \n{a + b}")


This is exactly the scenario in the vectorized dense layer built above: a 1-D bias vector `b` is broadcast across every row of an `(m, j)` activation matrix.

---

### Ideas to keep exploring
- Try a **harder digit pair** (e.g. `4` vs `9`, or `1` vs `7`) by changing the `mask` line in section 2.2.
- Add a **train/test split** (`sklearn.model_selection.train_test_split`) to measure generalization instead of training accuracy.
- Try **ReLU** activations for the hidden layers instead of sigmoid, and compare training speed/accuracy.
- Extend this into a **multi-class classifier** (all 10 digits, softmax output) — the natural next step after this binary lab.
